## OFFSET 分页
页数越深，OFFSET 越大，浪费越明显。PostgreSQL 官方文档也指出：被 OFFSET 跳过的行仍需在服务器内部计算。

所以offset实际上会处理offset的条数去丢弃。

## 游标分页
```sql
-- 第一页
SELECT id, title, created_at
FROM posts
ORDER BY created_at DESC, id DESC
LIMIT 20;
```
假设上述最后一条记录：

created_at = '2026-10-05 10:00:00'
id = 500

下一页要找的是“排在它后面”的记录：
```sql
SELECT id, title, created_at
FROM posts
WHERE (created_at, id) < ('2026-10-05 10:00:00', 500)
ORDER BY created_at DESC, id DESC
LIMIT 20;
```

如果有匹配排序和筛选条件的索引，数据库更容易从游标位置继续扫描，而不是反复处理前面所有记录。
```sql
CREATE INDEX idx_posts_created_at_id
ON posts (created_at DESC, id DESC);
```

- 适合：动态信息流、聊天消息、订单流水、日志流
- 不太适合：用户必须跳到“第 526 页”的后台报表

## 慢查询先看什么
```
接口慢
  ↓
确认耗时在哪：应用、数据库、外部 API、网络？
  ↓
拿到真实 SQL、真实参数、实际耗时
  ↓
EXPLAIN ANALYZE
  ↓
看扫描、过滤、排序、循环次数、估算与实际行数
  ↓
再决定：改 SQL、加/改索引、更新统计信息，还是接受当前计划
```
- EXPLAIN：看计划预测
- EXPLAIN ANALYZE：看实际执行结果

对 UPDATE、DELETE 这种有副作用的语句，线上不要随便直接执行 EXPLAIN ANALYZE。


### 最先看四件事

**1. Scan（扫描方式）**
- 含义：怎么找数据
- 常见信号：出现 `Seq Scan`（全表顺序扫描）

**2. Filter（过滤条件）**
- 含义：找到后又丢掉哪些行
- 常见信号：扫很多行、返回很少行，往往是索引不匹配

**3. Sort（排序）**
- 含义：是否额外排序
- 常见信号：大数据量排序很贵

**4. rows（行数预估）**
- 含义：预估和实际是否差很大
- 常见信号：差很大通常意味着统计信息不准，或条件相关性复杂

## 扫描方式与索引是否合理
### Seq Scan
`Seq Scan on orders`
顺序扫描整张表，下述情况合理：
- 查询需要返回表中大多数数据
- 表本来就很小
- 没有合适索引

### Index Scan
```sql
Index Scan using idx_orders_user_id on orders
  Index Cond: (user_id = 10)
```
先在索引里找 user_id = 10 的位置，再取对应行。

### Bitmap Heap Scan
```sql
Bitmap Heap Scan on orders
  Recheck Cond: (status = 'paid')
  -> Bitmap Index Scan on idx_orders_status
       Index Cond: (status = 'paid')
```
先用索引找出很多候选行的位置 → 再把位置整理成适合批量读取的顺序 → 回表读取真实数据